In [ ]:
!pip install langchain langchain-community langchain-groq faiss-cpu sentence-transformers PyPDF2 pypdf --q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install fpdf --q

  Preparing metadata (setup.py) ... done


In [ ]:
from fpdf import FPDF
import os

pdf=FPDF()
pdf.add_page()
pdf.set_font("Arial",size=12)

content = [
  "Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines and software capable of performing tasks that typically require human intelligence. These tasks include learning from data, recognizing patterns, understanding language, solving problems, and making decisions. AI systems use algorithms and large amounts of data to improve their performance over time, making them increasingly effective in various applications.",

"One of the most common uses of AI today is in everyday technology. Virtual assistants, recommendation systems, search engines, and language translation tools all rely on AI to provide personalized and efficient services. In industries such as healthcare, AI helps doctors analyze medical images, predict diseases, and improve patient care. In finance, AI is used to detect fraud, assess risks, and automate routine tasks.",

"AI is also transforming the way businesses operate. Companies use AI-powered tools to analyze customer behavior, optimize supply chains, and enhance productivity. Machine learning, a subset of AI, enables systems to learn from experience without being explicitly programmed. This capability allows businesses to make data-driven decisions and develop innovative products and services that meet customer needs more effectively.",

"Despite its many benefits, AI also presents challenges and ethical concerns. Issues such as data privacy, bias in algorithms, job displacement, and the responsible use of AI technologies require careful consideration. As AI continues to advance, governments, organizations, and researchers must work together to ensure that these technologies are developed and used in a way that benefits society while minimizing potential risks."
]
for line in content:
  pdf.multi_cell(0,10,line)

os.makedirs("/content/drive/MyDrive/Colab Notebooks/Datasets/pdfs",exist_ok=True)
pdf_path="/content/drive/MyDrive/Colab Notebooks/Datasets/pdfs/ragg.pdf"
pdf.output(pdf_path)

print("Created file")

Created file


In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"]="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"

#Step1:Load pdf
loader=PyPDFLoader("/content/drive/MyDrive/Colab Notebooks/Datasets/pdfs/ragg.pdf")
pages=loader.load()

splitter=CharacterTextSplitter(chunk_size=500,chunk_overlap=50)
documents=splitter.split_documents(pages)

#step3:Embdediiing +FAISS
embedding=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

db=FAISS.from_documents(documents,embedding)


llm=ChatGroq(model="llama-3.1-8b-instant")
prompt=PromptTemplate.from_template(
    "Use the following context to answer the question:\n\nContext:\n{Context}\n\nQuestion:\n{question}\nAnswer:"
)
parser=StrOutputParser()

def ask_pdf(query:str):
  docs=db.similarity_search_with_score(query,k=3)
  Context="\n\n".join([d[0].page_content for d in docs])
  chain=prompt|llm|parser
  return chain.invoke({"Context":Context,"question":query})

while True:
  query=input("Ask a question: ")
  if query.lower()=="exit":
    break
  answer=ask_pdf(query)
  print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ask a question: exit


In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

os.environ["GROQ_API_KEY"]="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"

llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0)

prompt=PromptTemplate(
    input_variables=["instruction"],
    template="Follow the instruction and provide output clearly:\n\nInstruction:{instruction}\n\nAnswer:"
)
parser=StrOutputParser()
chain=prompt |llm|parser

response=chain.invoke({
    "instruction":"You are Solver which give code in python for User need in Streamlit   'calculator'"
})

print(response)

**Streamlit Calculator Code**

Below is a simple implementation of a calculator using Streamlit in Python:

```python
import streamlit as st

# Create a Streamlit app
st.title("Calculator")

# Create input fields for numbers
num1 = st.number_input("Enter first number")
num2 = st.number_input("Enter second number")

# Create a selectbox for operations
operation = st.selectbox("Select operation", ["Addition", "Subtraction", "Multiplication", "Division"])

# Create a function to perform calculations
def calculate(num1, num2, operation):
    if operation == "Addition":
        return num1 + num2
    elif operation == "Subtraction":
        return num1 - num2
    elif operation == "Multiplication":
        return num1 * num2
    elif operation == "Division":
        if num2 != 0:
            return num1 / num2
        else:
            return "Error: Division by zero"

# Create a button to calculate
if st.button("Calculate"):
    result = calculate(num1, num2, operation)
    st.write("Resul